In [1]:
from pyspark.sql import SparkSession

def calculate_red_violations(data_source, output_uri):
    """Processa as inspeções, identifica os 10 estabelecimentos com mais infrações RED e salva o resultado no S3."""

    # Criar ou reutilizar uma sessão Spark
    spark = SparkSession.builder.appName("Food Establishment - RED Violations").getOrCreate()

    # Ler o arquivo CSV do S3 e criar um DataFrame
    restaurants_df = spark.read.option("header", "true").option("inferSchema", "true").csv(data_source)

    # Criar uma visão temporária para realizar consultas com Spark SQL
    restaurants_df.createOrReplaceTempView("restaurant_violations")

    # Filtrar as infrações RED, agrupar por estabelecimento e obter o Top 10
    result_df = spark.sql("""
        SELECT name, COUNT(*) AS total_red_violations
        FROM restaurant_violations
        WHERE violation_type = 'RED'
        GROUP BY name
        ORDER BY total_red_violations DESC
        LIMIT 10
    """)

    # Exibir os resultados
    print("Top 10 establishments with the most RED violations:")
    result_df.show(truncate=False)

    # Salvar os resultados no S3 em formato CSV
    result_df.write.option("header", "true").mode("overwrite").csv(output_uri)

    # Retornar o DataFrame com os resultados
    return result_df

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1792397246011_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
# Definir as rotas de entrada e saída no Amazon S3
data_source = "s3://bucket-rosa-lab1/food_establishment_data.csv"
output_uri = "s3://bucket-rosa-lab1/output/red_violations/"

# Executar a função
result_df = calculate_red_violations(data_source, output_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Top 10 establishments with the most RED violations:
+----------------------+--------------------+
|name                  |total_red_violations|
+----------------------+--------------------+
|SUBWAY                |322                 |
|T-MOBILE PARK         |315                 |
|WHOLE FOODS MARKET    |299                 |
|PCC COMMUNITY MARKETS |251                 |
|TACO TIME             |240                 |
|MCDONALD'S            |177                 |
|THAI GINGER           |153                 |
|SAFEWAY INC #1508     |143                 |
|TAQUERIA EL RINCONSITO|134                 |
|HIMITSU TERIYAKI      |128                 |
+----------------------+--------------------+